# Exercise 4: Transformers on Images + GLU-MLP Ablations (ViT × GLU Variants)

## In this exercise you will combine two influential ideas:

Vision Transformers (ViT) from “An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale” (Dosovitskiy et al., 2020) https://arxiv.org/pdf/2010.11929:
ViT shows that you can treat an image like a sequence of tokens by splitting it into non-overlapping patches (e.g. 16×16 in the paper), embedding each patch into a vector, adding positional information, and then applying standard Transformer blocks for classification.

Gated MLPs (GLU variants) from “GLU Variants Improve Transformer” (Shazeer, 2020) https://arxiv.org/pdf/2002.05202:
Shazeer proposes replacing the standard Transformer feed-forward layer (FFN/MLP) with gated linear unit (GLU) variants such as GEGLU and SwiGLU, which often improves training dynamics and final performance under comparable compute/parameter budgets.

## What you will do

You will implement a tiny ViT-style classifier for MNIST, then run a controlled ablation where you replace the MLP inside each Transformer block:

Baseline FFN (GELU):
Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)

GLU-family MLPs (choose at least two and justify):

GEGLU, SwiGLU, other activation functions

Your goal is to evaluate whether these GLU variants change:

- convergence speed (loss vs steps),

- final test accuracy,

- and/or stability across runs.

## Key ViT concepts you will implement

- To convert MNIST images into Transformer tokens, you will:
  Patchify each 28×28 image into non-overlapping P×P patches.
  If P=4, then you get a 7×7 patch grid → 49 tokens per image.

- Embed patches with a linear layer: patch vectors → d_model.

- Add positional embeddings so the model knows where each patch came from.

- Apply n_layers Transformer encoder blocks.

- Pool token features (e.g., mean pooling) and project to 10 classes.

## Key GLU concept you will implement

GLU-style MLPs replace a standard FFN with a gating mechanism:
compute two projections a and b, apply a nonlinearity to a (variant-dependent), multiply elementwise: act(a) * b, project back to d_model.
To keep the comparison fair, use the 2/3 width rule from Shazeer.

What we provide vs what you implement

### We provide:

- MNIST loading + dataloaders

- a minimal training loop structure (AdamW)

- a suggested small model configuration that runs on CPU

### You implement:

- patch tokenization (patchify)

- patch embedding + positional embedding strategy

- a pre-LN Transformer encoder block using nn.MultiheadAttention

- at least two GLU MLP variants + one FFN baseline

- metric logging sufficient to support your conclusion

## Deliverables

Run at least 3 variants (baseline + the activation functions you choose for GLU) and report:

- final and best test accuracy

- number of trainable parameters

- a plot or printed summary of loss/accuracy over epochs

- a short discussion of your results

In [25]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [26]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Convert images to patch tokens."""
    # TODO: Implement a tokenization strategy
    B, C, H, W = x.shape # after checking below, we know we have [batch size, channels, height, width]
    if H % patch_size != 0:
        raise ValueError("Image size must be divisible by patch_size")
    grid_size = int(H / patch_size)
    
    x = x.reshape(B, C, grid_size, patch_size, grid_size, patch_size)
    # want B, C, H, W -> grid_size, grid_size, B, C, patch_size, patch_size
    return x.permute(0, 1, 2, 4, 3, 5)
    
tfm = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
train_loader = DataLoader(train_ds, batch_size=3, shuffle=True, num_workers=0)

x_original = next(iter(train_loader))
print(x_original[0].shape)
x = patchify(x_original[0], 14)
print(x.shape)

transforms.ToPILImage()(x[0, 0, 1, 1, :, :])


    


torch.Size([3, 1, 28, 28])
torch.Size([3, 1, 2, 2, 14, 14])


In [27]:
transforms.ToPILImage()(x_original[0][0])

In [28]:
# TODO: Add positional encoding as done in the ViT paper and patch projection
class PatchEmbed(nn.Module):
    def __init__(self, patch_dim: int, d_model: int):
        super().__init__()
        # TODO: implement
        self.layer = nn.Linear(in_features=patch_dim, out_features=d_model)
        

    def forward(self, x_patches: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        x_patches = x_patches.flatten(-2)
        return self.layer(x_patches)


class PositionalEmbedding(nn.Module):
    def __init__(self, num_tokens: int, d_model: int):
        super().__init__()
        # TODO: implement
        self.pos_embedding = nn.Parameter(torch.zeros(1, num_tokens, d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        
        
        return x.flatten(-3,-2) + self.pos_embedding

In [29]:
# TODO: Define the variants you want to compare against each other from the GLU paper. Justify your choice.
class FeedForward(nn.Module):
    """
    Standard Transformer FFN:
      x -> Linear(d_model->d_ff) -> GELU -> Dropout -> Linear(d_ff->d_model) -> Dropout
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: implement
        self.layers = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        return self.layers(x)


class GLUFeedForward(nn.Module):
    """GLU-family FFN"""
    def __init__(self, d_model: int, d_ff_gated: int, dropout: float, variant: str):
        super().__init__()
        # TODO: implement
        # we compare to GELU and SwiGLU as they had best performance in perplexity in the GLU paper
        if variant == "GEGLU":
            self.activation = nn.GELU()
        elif variant == "SwiGLU":
            self.activation = nn.SiLU()

        self.layers1 = nn.Sequential(
            nn.Linear(d_model, d_ff_gated * 2), # *2 because we're going to split the output in half for GLU
        )
        self.layers2 = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_ff_gated, d_model),
            nn.Dropout(dropout)
        )
        self.variant = variant



    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        
        x = self.layers1(x)
        x_gated, x_linear = x.chunk(2, dim=-1) # split in half for GLU
        x_gated = self.activation(x_gated)
        x = x_gated * x_linear # element-wise multiplication for GLU
        x = self.layers2(x)
        return x

In [30]:
class TransformerEncoderBlock(nn.Module):
    """
    Pre-LN encoder block:
      x = x + Dropout(SelfAttn(LN(x)))
      x = x + Dropout(MLP(LN(x)))
    """
    def __init__(self, d_model: int, n_heads: int, mlp: nn.Module, dropout: float):
        super().__init__()
        # TODO: implement. For attention use nn.MultiHeadAttention 

        # we follow paper structure
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=dropout)
        self.mlp = mlp

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        attn_out, _ = self.self_attn(self.ln1(x), self.ln1(x), self.ln1(x))
        x = attn_out + x
        x = self.mlp(self.ln1(x)) + x
        return x

In [31]:
class TinyViT(nn.Module):
    """
    Tiny ViT-style classifier for MNIST.
    - patchify -> patch embed -> pos embed -> blocks -> mean pool -> head
    """
    def __init__(
        self,
        patch_size: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        dropout: float,
        mlp_kind: str,
    ):
        super().__init__()
        assert 28 % patch_size == 0
        grid = 28 // patch_size
        self.num_tokens = grid * grid
        self.patch_size = patch_size
        patch_dim = patch_size * patch_size

        # TODO: implement a strategy for embedding the patches
        self.patch_embed = PatchEmbed(patch_dim=patch_dim, d_model=d_model)
        self.pos_embed = PositionalEmbedding(num_tokens=self.num_tokens, d_model=d_model)

        # TODO: implement a strategy to select the right mlp version for your experiment
        if mlp_kind == "baseline":
            mlp = FeedForward(d_model=d_model, d_ff=d_ff, dropout=dropout)
        elif mlp_kind == "GEGLU":
            mlp = GLUFeedForward(d_model=d_model, d_ff_gated=d_ff, dropout=dropout, variant="GEGLU")
        elif mlp_kind == "SwiGLU":
            mlp = GLUFeedForward(d_model=d_model, d_ff_gated=d_ff, dropout=dropout, variant="SwiGLU")

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                mlp=mlp, # TODO: Feed your mlp to the encoder blocks
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

        # TODO: Add a head to project to the amount of output classes you have
        self.head = nn.Linear(d_model, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Implement
        x = patchify(x, self.patch_size)
        # print(x.shape)
        # print(x.flatten(-2).shape)
        
        x = self.patch_embed(x)
        # print(x.shape)
        x = self.pos_embed(x)
        # print(x.shape)
        
        x = x.squeeze()
        # print(x.shape)
        for block in self.blocks:
            x = block(x)
        x = x.mean(dim=1) # mean pooling
        logits = self.head(x)


        return x

patch_size = 4
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
dropout = 0.1
model_debug = TinyViT(
    patch_size=patch_size,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    d_ff=d_ff,
    dropout=dropout,
    mlp_kind="baseline",
)

input = torch.randn(2, 1, 28, 28)
output = model_debug(input)


In [32]:
@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    batch_size: int = 128
    epochs: int = 20
    lr: float = 2e-4
    weight_decay: float = 0.01
    device: str = "cpu"  # set "cuda" if available

In [33]:
def train_one_run(
    mlp_kind: str,
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    cfg: TrainConfig,
) -> dict:
    model.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_losses: list[float] = []
    test_accs: list[float] = []

    for epoch in range(cfg.epochs):

        # Train loop
        model.train()
        for i, (xb, yb) in enumerate(train_loader):
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            logits = model(xb)
            loss = F.cross_entropy(logits, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_losses.append(loss.item())

        # Evaluation loop NOTE: Should be no need to change this
        model.eval()
        correct = 0.0
        total = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                logits = model(xb)
                correct += (logits.argmax(dim=-1) == yb).float().sum().item()
                total += yb.numel()

        test_accs.append(correct / total)
        print(f"[{mlp_kind}] epoch {epoch+1}/{cfg.epochs} | test acc: {test_accs[-1]:.4f}| train loss: {sum(train_losses[-len(train_loader):])/len(train_loader):.4f}")

    return {
        # TODO: Return your metrics that you think will support your claim for this experiment
        "losses": train_losses,
        "test_accs": test_accs,
    }

In [34]:
# cfg = TrainConfig(seed=0, batch_size=128, epochs=20, lr=1e-4, weight_decay=0.01, device="mps")

tfm = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=0)

print("shape of batch:", next(iter(train_loader))[0].shape)

# Tiny model example. TODO: You're welcome to experiment with these parameters
patch_size = 4
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
dropout = 0.1

reps = 5 # for statistical significance we run the test multiple times each

runs = ["baseline", "GEGLU", "SwiGLU"] # TODO: Name your runs
cfgs = [TrainConfig(seed=i, batch_size=128, epochs=20, lr=2e-4, weight_decay=0.01, device="mps") for i in range(5)] # TODO: You can also experiment with different configs for each run if you want
results = torch.zeros((2, len(runs), len(cfgs))) 

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)

for i, kind in enumerate(runs):
    for j, cfg in enumerate(cfgs):
        set_seed(cfg.seed)
    

        model = TinyViT(
            patch_size=patch_size,
            d_model=d_model,
            n_heads=n_heads,
            n_layers=n_layers,
            d_ff=d_ff,
            dropout=dropout,
            mlp_kind=kind,
        )
        # TODO: print anything you might want here
        print(f"\nRun: {kind} | cfg: {cfg} | " )
        out = train_one_run(kind, model, train_loader, test_loader, cfg)
        results[0, i, j] = out["losses"][-1]
        results[1, i, j] = out["test_accs"][-1]

    
    




shape of batch: torch.Size([128, 1, 28, 28])

Run: baseline | cfg: TrainConfig(seed=0, batch_size=128, epochs=20, lr=0.0002, weight_decay=0.01, device='mps') | 
[baseline] epoch 1/20 | test acc: 0.3218| train loss: 2.2298
[baseline] epoch 2/20 | test acc: 0.4289| train loss: 1.7773
[baseline] epoch 3/20 | test acc: 0.6258| train loss: 1.3839
[baseline] epoch 4/20 | test acc: 0.6875| train loss: 1.0349
[baseline] epoch 5/20 | test acc: 0.7480| train loss: 0.8766
[baseline] epoch 6/20 | test acc: 0.7893| train loss: 0.7392
[baseline] epoch 7/20 | test acc: 0.8079| train loss: 0.6425
[baseline] epoch 8/20 | test acc: 0.8404| train loss: 0.5663
[baseline] epoch 9/20 | test acc: 0.8600| train loss: 0.5046
[baseline] epoch 10/20 | test acc: 0.8707| train loss: 0.4569
[baseline] epoch 11/20 | test acc: 0.8729| train loss: 0.4115
[baseline] epoch 12/20 | test acc: 0.8962| train loss: 0.3843
[baseline] epoch 13/20 | test acc: 0.9001| train loss: 0.3590
[baseline] epoch 14/20 | test acc: 0.9027|

In [ ]:
# from scipy import stats


# t_stat_GEGLU, p_value_GEGLU = stats.ttest_ind(results[1, 0, :], results[1, 1, :])
# print(f"P-value: {p_value_GEGLU:.4f}")
# t_stat_SwiGLU, p_value_SwiGLU = stats.ttest_ind(results[1, 0, :], results[1, 2, :])
# print(f"P-value: {p_value_SwiGLU:.4f}")
# t_stat_GEGLU_SWiGLU, p_value_GEGLU_SwiGLU = stats.ttest_ind(results[1, 1, :], results[1, 2, :])
# print(f"P-value: {p_value_GEGLU_SwiGLU:.4f}")

P-value: 0.0001
P-value: 0.0002
P-value: 0.0018


torch.Size([2, 3, 5])